# Track-A Experiment 7: QLoRA Fine-tuning of Gemma 3 4B

**Task**: SemEval-2026 Task 4 — Narrative Story Similarity (Track A)  
**Approach**: QLoRA fine-tuning for binary classification (text_a_is_closer: true/false)  
**Model**: google/gemma-3-4b-it  

## Setup
- Model loaded from HuggingFace Hub using Colab secrets token
- Data pulled from your HuggingFace dataset repo
- Fine-tuned model + results pushed back to HuggingFace

## 0. Configuration

**IMPORTANT**: Fill in your HuggingFace details below before running.

In [1]:
# ════════════════════════════════════════════════════════════════════════════
# FILL THESE IN
# ════════════════════════════════════════════════════════════════════════════

HF_TRAIN_REPO = "Chandan683/task4_syn_data"       # 1900 synthetic training examples
HF_DEV_REPO = "Chandan683/Dev_set_data"            # 200 validation examples
HF_TEST_REPO = "Chandan683/task4_test_data"        # 400 test examples
HF_MODEL_OUTPUT_REPO = "Chandan683/gemma3-4b-narrative-similarity"  # where to push fine-tuned model
HF_RESULTS_REPO = "Chandan683/semeval-task4-results"  # where to push results

# Model
BASE_MODEL = "google/gemma-3-1b-it"

# Training hyperparameters (tuned for ~1900 training examples on 4B model)
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 16
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## 1. Install Dependencies

In [2]:
!pip install -q transformers>=4.44.0 datasets accelerate peft bitsandbytes trl huggingface_hub scikit-learn

## 2. Authenticate with HuggingFace

In [3]:
from google.colab import userdata
from huggingface_hub import login

# Reads the HF_TOKEN from Colab secrets
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


## 3. Load Data from HuggingFace

In [4]:
from datasets import load_dataset

# Load training data (synthetic, 1900 examples)
train_dataset = load_dataset(HF_TRAIN_REPO)
print("Train dataset:", train_dataset)

# Load dev/validation data (200 examples)
dev_dataset = load_dataset(HF_DEV_REPO)
print("\nDev dataset:", dev_dataset)

# Load test data (400 examples, no labels)
test_dataset = load_dataset(HF_TEST_REPO)
print("\nTest dataset:", test_dataset)

print(f"\nTrain split: {train_dataset['train'].num_rows} examples")
print(f"Dev split:   {dev_dataset['validation'].num_rows} examples")
print(f"Test split:  {test_dataset['test'].num_rows} examples")

# Preview one training example
print("\n--- Sample training example ---")
print(train_dataset["train"][0])

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

synthetic_data_for_classification.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1900 [00:00<?, ? examples/s]

Train dataset: DatasetDict({
    train: Dataset({
        features: ['model_name', 'anchor_text', 'text_a', 'text_b', 'text_a_is_closer'],
        num_rows: 1900
    })
})


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

...


Dev dataset: DatasetDict({
    validation: Dataset({
        features: ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer'],
        num_rows: 200
    })
})


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

test_track_a.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]


Test dataset: DatasetDict({
    test: Dataset({
        features: ['anchor_text', 'text_a', 'text_b'],
        num_rows: 400
    })
})

Train split: 1900 examples
Dev split:   200 examples
Test split:  400 examples

--- Sample training example ---
{'model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo', 'anchor_text': 'A mysterious individual, known only by their alias "The Wanderer", arrives in the small, rural town of Willow Creek, sparking curiosity and intrigue among its residents. The Wanderer, a woman with no discernible past or motivations, settles into a local boarding house and begins to integrate into the community. As the townspeople grow accustomed to her presence, they start to notice a series of subtle yet significant changes: abandoned buildings are repaired, neglected gardens are tended, and long-forgotten traditions are revived. The Wanderer\'s actions are met with a mix of gratitude and suspicion, with some viewing her as a benevolent force and others as a poten

## 4. Format Data for Instruction Tuning

Convert each triple into a chat-format prompt with the label as the response.

In [5]:
SYSTEM_PROMPT = """You are an expert in narrative analysis. Your task is to determine which of two stories is more narratively similar to an anchor story.

Narrative similarity is based on three components:
1. Abstract Theme: The underlying ideas and motives of the story.
2. Course of Action: The sequence of central events and turning points.
3. Outcomes: The results and conclusions of the story.

Do NOT focus on surface-level similarities like setting, time period, character names, or genre. Focus on the deeper narrative structure.

Respond with ONLY \"A\" or \"B\" (nothing else)."""


def format_as_chat(example):
    """Convert a Track-A example into chat messages for instruction tuning."""
    user_msg = (
        f"### Anchor Story\n{example['anchor_text']}\n\n"
        f"### Text A\n{example['text_a']}\n\n"
        f"### Text B\n{example['text_b']}\n\n"
        f"Which text is more narratively similar to the anchor?"
    )

    label = "A" if example["text_a_is_closer"] else "B"

    example["messages"] = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_msg},
        {"role": "assistant", "content": label},
    ]
    return example


def format_as_chat_inference(example):
    """Convert a Track-A example into chat messages for inference (no label)."""
    user_msg = (
        f"### Anchor Story\n{example['anchor_text']}\n\n"
        f"### Text A\n{example['text_a']}\n\n"
        f"### Text B\n{example['text_b']}\n\n"
        f"Which text is more narratively similar to the anchor?"
    )

    example["messages"] = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_msg},
    ]
    return example


# Filter out rows with null text fields from synthetic training data
train_raw = train_dataset["train"].filter(
    lambda x: x["anchor_text"] is not None and x["text_a"] is not None and x["text_b"] is not None
)
train_data = train_raw.map(format_as_chat)

# Dev data (full 200 examples with labels)
dev_data = dev_dataset["validation"]

# Test data (no labels)
test_data = test_dataset["test"].map(format_as_chat_inference)

print(f"Training examples: {len(train_data)} (after filtering nulls from {train_dataset['train'].num_rows})")
print(f"Dev examples:      {len(dev_data)}")
print(f"Test examples:     {len(test_data)}")
print(f"\n--- Formatted training example ---")
print(train_data[0]["messages"])

Filter:   0%|          | 0/1900 [00:00<?, ? examples/s]

Map:   0%|          | 0/1897 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Training examples: 1897 (after filtering nulls from 1900)
Dev examples:      200
Test examples:     400

--- Formatted training example ---
[{'content': 'You are an expert in narrative analysis. Your task is to determine which of two stories is more narratively similar to an anchor story.\n\nNarrative similarity is based on three components:\n1. Abstract Theme: The underlying ideas and motives of the story.\n2. Course of Action: The sequence of central events and turning points.\n3. Outcomes: The results and conclusions of the story.\n\nDo NOT focus on surface-level similarities like setting, time period, character names, or genre. Focus on the deeper narrative structure.\n\nRespond with ONLY "A" or "B" (nothing else).\n\n### Anchor Story\nA mysterious individual, known only by their alias "The Wanderer", arrives in the small, rural town of Willow Creek, sparking curiosity and intrigue among its residents. The Wanderer, a woman with no discernible past or motivations, settles into a lo

## 5. Load Model with QLoRA (4-bit quantization)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=hf_token,
)
tokenizer.padding_side = "right"

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {BASE_MODEL}")
print(f"Model dtype: {model.dtype}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded: google/gemma-3-1b-it
Model dtype: torch.float32
GPU memory used: 1.6 GB


## 6. Configure LoRA

In [7]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


## 7. Train with SFTTrainer

In [8]:
from trl import SFTTrainer, SFTConfig

tokenizer.model_max_length = MAX_SEQ_LENGTH

training_args = SFTConfig(
    output_dir="./results_gemma3_qlora",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Tokenizing train dataset:   0%|          | 0/1897 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1897 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Starting training...


Step,Training Loss
10,2.559600
20,1.833200
30,1.402800
40,1.276600
50,1.234100
60,1.208900
70,1.142700
80,1.164100
90,1.145100
100,1.082700


Training complete!


## 8. Evaluate on Dev Set

In [9]:
from sklearn.metrics import accuracy_score, classification_report
import json


def predict_single(messages, model, tokenizer):
    """Run inference on a single example and return predicted label."""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip().upper()

    if response.startswith("A"):
        return True
    elif response.startswith("B"):
        return False
    else:
        print(f"  Unparseable: '{response}', defaulting to A")
        return True  # fallback


# Evaluate on full dev set (Chandan683/Dev_set_data, 200 examples)
dev_eval = dev_data.map(format_as_chat_inference)

gold_labels = []
pred_labels = []
predictions = []

print(f"Evaluating on {len(dev_eval)} dev examples...")
for i, example in enumerate(dev_eval):
    pred = predict_single(example["messages"], model, tokenizer)
    gold = dev_data[i]["text_a_is_closer"]

    gold_labels.append(gold)
    pred_labels.append(pred)
    predictions.append({
        "id": i,
        "gold": gold,
        "pred": pred,
        "correct": gold == pred,
    })

    if (i + 1) % 20 == 0:
        current_acc = sum(p["correct"] for p in predictions) / len(predictions)
        print(f"  [{i+1}/{len(dev_eval)}] running accuracy: {current_acc:.4f}")

accuracy = accuracy_score(gold_labels, pred_labels)
print(f"\n{'='*60}")
print(f"Dev Accuracy: {accuracy:.4f} ({sum(g == p for g, p in zip(gold_labels, pred_labels))}/{len(gold_labels)})")
print(f"\nClassification Report:")
print(classification_report(gold_labels, pred_labels, target_names=["B (text_b)", "A (text_a)"]))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.


Evaluating on 200 dev examples...
  [20/200] running accuracy: 0.3000
  [40/200] running accuracy: 0.3000
  [60/200] running accuracy: 0.4000
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [80/200] running accuracy: 0.4750
  [100/200] running accuracy: 0.4400
  [120/200] running accuracy: 0.4333
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [140/200] running accuracy: 0.4571
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [160/200] running accuracy: 0.4500
  [180/200] running accuracy: 0.4444
  [200/200] running accuracy: 0.4350

Dev Accuracy: 0.4350 (87/200)

Classification Report:
              precision    recall  f1-score   support

  B (text_b)       0.45      0.67      0.54        99
  A (text_a)       0.39

## 9. Generate Test Predictions

In [10]:
test_predictions = []

print(f"Generating predictions for {len(test_data)} test examples...")
for i, example in enumerate(test_data):
    pred = predict_single(example["messages"], model, tokenizer)
    test_predictions.append({"text_a_is_closer": pred})

    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{len(test_data)}] processed.")

# Save as JSONL (submission format)
with open("track_a.jsonl", "w") as f:
    for pred in test_predictions:
        f.write(json.dumps(pred) + "\n")

print(f"\nTest predictions saved to track_a.jsonl")
print(f"Total predictions: {len(test_predictions)}")
print(f"Label distribution: {sum(p['text_a_is_closer'] for p in test_predictions)} \u00d7 A, "
      f"{sum(not p['text_a_is_closer'] for p in test_predictions)} \u00d7 B")

Generating predictions for 400 test examples...
  Unparseable: '11111111', defaulting to A
  [20/400] processed.
  [40/400] processed.
  [60/400] processed.
  [80/400] processed.
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [100/400] processed.
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [120/400] processed.
  [140/400] processed.
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [160/400] processed.
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [180/400] processed.
  [200/400] processed.
  [220/400] processed.
  [240/400] processed.
  Unparseable: 'THETHETHETHETHETHETHETHE', defaulting to A
  [260/400] processed.
  [280/400] processed.
  [300/400] processed.
  [320/400] processed.
  [340/400] processed.
  [360/400] processed.
  [380/400] processed.
  [400/400] processed.

Test predictions saved to track_a.jsonl
Total predictions: 400
Label distribution: 105 × A, 295 × B


## 10. Save Dev Evaluation Results

In [11]:
results = {
    "experiment": "track_a_exp7_qlora_finetune_gemma3",
    "base_model": BASE_MODEL,
    "method": "QLoRA",
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dev_accuracy": accuracy,
    "dev_correct": int(sum(g == p for g, p in zip(gold_labels, pred_labels))),
    "dev_total": len(gold_labels),
}

with open("results_exp7.json", "w") as f:
    json.dump(results, f, indent=2)

# Save per-example predictions
with open("dev_predictions.jsonl", "w") as f:
    for pred in predictions:
        f.write(json.dumps(pred) + "\n")

print("Results saved locally:")
print("  - results_exp7.json")
print("  - dev_predictions.jsonl")
print("  - track_a.jsonl")

Results saved locally:
  - results_exp7.json
  - dev_predictions.jsonl
  - track_a.jsonl


## 11. Push Fine-tuned Model to HuggingFace

In [12]:
# Save and push the LoRA adapter
model.save_pretrained("./gemma3_4b_lora_adapter")
tokenizer.save_pretrained("./gemma3_4b_lora_adapter")

model.push_to_hub(HF_MODEL_OUTPUT_REPO, token=hf_token)
tokenizer.push_to_hub(HF_MODEL_OUTPUT_REPO, token=hf_token)

print(f"Model pushed to: https://huggingface.co/{HF_MODEL_OUTPUT_REPO}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  615kB / 26.1MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p73qidbfk/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...mp73qidbfk/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to: https://huggingface.co/Chandan683/gemma3-4b-narrative-similarity


## 12. Push Results to HuggingFace

In [13]:
from huggingface_hub import HfApi

api = HfApi()

# Create the results repo if it doesn't exist
try:
    api.create_repo(HF_RESULTS_REPO, repo_type="dataset", exist_ok=True, token=hf_token)
except Exception as e:
    print(f"Repo creation note: {e}")

# Upload results files
for filename in ["results_exp7.json", "dev_predictions.jsonl", "track_a.jsonl"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"track_a_exp7_qlora_gemma3/{filename}",
        repo_id=HF_RESULTS_REPO,
        repo_type="dataset",
        token=hf_token,
    )
    print(f"  Uploaded: {filename}")

print(f"\nResults pushed to: https://huggingface.co/datasets/{HF_RESULTS_REPO}")

  Uploaded: results_exp7.json
  Uploaded: dev_predictions.jsonl
  Uploaded: track_a.jsonl

Results pushed to: https://huggingface.co/datasets/Chandan683/semeval-task4-results


## 13. Summary

In [14]:
print("=" * 60)
print("EXPERIMENT 7 SUMMARY")
print("=" * 60)
print(f"Base model:       {BASE_MODEL}")
print(f"Method:           QLoRA (4-bit NF4)")
print(f"LoRA rank:        {LORA_R}")
print(f"Training data:    {len(train_data)} examples (from {HF_TRAIN_REPO})")
print(f"Dev data:         {len(dev_data)} examples (from {HF_DEV_REPO})")
print(f"Test data:        {len(test_data)} examples (from {HF_TEST_REPO})")
print(f"Epochs:           {NUM_EPOCHS}")
print(f"Learning rate:    {LEARNING_RATE}")
print(f"Eff. batch size:  {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Dev accuracy:     {accuracy:.4f}")
print(f"Test predictions: {len(test_predictions)}")
print(f"\nModel:   https://huggingface.co/{HF_MODEL_OUTPUT_REPO}")
print(f"Results: https://huggingface.co/datasets/{HF_RESULTS_REPO}")

EXPERIMENT 7 SUMMARY
Base model:       google/gemma-3-1b-it
Method:           QLoRA (4-bit NF4)
LoRA rank:        16
Training data:    1897 examples (from Chandan683/task4_syn_data)
Dev data:         200 examples (from Chandan683/Dev_set_data)
Test data:        400 examples (from Chandan683/task4_test_data)
Epochs:           3
Learning rate:    0.0002
Eff. batch size:  16
Dev accuracy:     0.4350
Test predictions: 400

Model:   https://huggingface.co/Chandan683/gemma3-4b-narrative-similarity
Results: https://huggingface.co/datasets/Chandan683/semeval-task4-results
